In [91]:
import argparse
import concurrent
from dotenv import load_dotenv
from tqdm import tqdm
import textgrad as tg
from textgrad.tasks import load_task
import numpy as np
import random
import pandas as pd
import ast

from eval_instructions import *

load_dotenv(override=True)

np.random.seed(21)
random.seed(21)

In [2]:
def eval_sample(item, eval_fn, model):
    x, y = item
    x = tg.Variable(x, requires_grad=False, role_description="query to the language model")
    y = tg.Variable(y, requires_grad=False, role_description="correct answer for the query")
    response = model(x)
    try:
        eval_output_variable = eval_fn(inputs={"prediction": response, "ground_truth_answer": y})
        return int(eval_output_variable.value)
    except:
        eval_output_variable = eval_fn([x, y, response]) # Consider the query in the evaluation
        eval_output_parsed = eval_fn.parse_output(eval_output_variable)
        return int(eval_output_parsed)

def eval_dataset(test_set, eval_fn, model,  max_samples=None):
    if max_samples is None:
        max_samples = len(test_set)
    accuracy_list = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        futures = []
        for sample in test_set:
            future = executor.submit(eval_sample, sample, eval_fn, model)
            futures.append(future)
            if len(futures) >= max_samples:
                break
        tqdm_loader = tqdm(concurrent.futures.as_completed(futures), total=len(futures), position=0)
        for future in tqdm_loader:
            acc_item = future.result()
            accuracy_list.append(acc_item)
            tqdm_loader.set_description(f"Accuracy: {np.mean(accuracy_list)}")
    return accuracy_list

def run_validation_revert(system_prompt: tg.Variable, results, model, eval_fn, val_set):
    val_performance = np.mean(eval_dataset(val_set, eval_fn, model))
    previous_performance = np.mean(results["validation_acc"][-1])
    print("val_performance: ", val_performance)
    print("previous_performance: ", previous_performance)
    previous_prompt = results["prompt"][-1]

    if val_performance < previous_performance:
        print(f"Rejected prompt: {system_prompt.value}")
        system_prompt.set_value(previous_prompt)
        val_performance = previous_performance
    
    results["validation_acc"].append(val_performance)

In [3]:
llm_api_eval = tg.get_engine("experimental:gpt-4.1", cache=False)
llm_api_test = tg.get_engine("experimental:gpt-4.1", cache=False)
tg.set_backward_engine(llm_api_eval, override=True)

# # Load the data and the evaluation function
# train_set, val_set, test_set, eval_fn = load_task("text_grad_data", evaluation_api=llm_api_eval) # TODO add in your own data
# print("Train/Val/Test Set Lengths: ", len(train_set), len(val_set), len(test_set))
# STARTING_SYSTEM_PROMPT = ""
# print("Starting System Prompt: \n", STARTING_SYSTEM_PROMPT)

# train_loader = tg.tasks.DataLoader(train_set, batch_size=3, shuffle=True)

# New Idea


In [4]:


initial_solution = """To solve the equation 3x^2 - 7x + 2 = 0, we use the quadratic formula:
x = (-b ± √(b^2 - 4ac)) / 2a
a = 3, b = -7, c = 2
x = (7 ± √((-7)^2 + 4(3)(2))) / 6
x = (7 ± √73) / 6
The solutions are:
x1 = (7 + √73)
x2 = (7 - √73)"""

outer_prompt = tg.Variable("You are a math expert. You will be asked to solve a math problem. Solve it accurately and concisely.",
                            requires_grad=True,
                            role_description="system prompt")

solution = tg.Variable(initial_solution,
                       requires_grad=False,
                       role_description="solution to the math question")

loss_system_prompt = tg.Variable("""You will evaluate a solution to a math question. 
Do not attempt to solve it yourself, do not give a solution, only identify errors. Be super concise.""",
                                 requires_grad=False,
                                 role_description="system prompt")
                              
loss_fn = tg.TextLoss(loss_system_prompt)
optimizer = tg.TGD([outer_prompt, solution])

loss = loss_fn(solution)
print("TextGrad Loss: \n", loss)

loss.backward()
optimizer.step()
print("Optimized System Prompt: \n", outer_prompt.value)

TextGrad Loss: 
 Errors:
1. In the quadratic formula, the coefficients are a = 3, b = -7, c = 2, but you used b = -7 without proper sign substitution in your formula.
2. The discriminant was calculated incorrectly: it should be (-7)^2 - 4(3)(2), not (-7)^2 + 4(3)(2).
3. The denominator in the quadratic formula is 2a = 6, which is correct, but the numerators in final answers omit dividing by 6.
4. Final solutions lack the division by 6: x1 = (7 + √73) / 6 and x2 = (7 - √73) / 6.

Summary: Discriminant calculation sign is incorrect; missing division by 6 in final solutions.
Optimized System Prompt: 
 You are a mathematics expert. When presented with a math problem, carefully analyze it and provide a clear, accurate, and step-by-step solution. Ensure your explanation is concise and easy to understand.


In [ ]:
# # Quoted OP generation
# quoted_op_prompt = "Respond to the following query using word-for-word quotes from the sources provided below. Clearly indicate the quotes with quotation marks to avoid plagiarizing! Be concise in your response and focus on information that responds to the query. Do not refer to the sources in your response. In your response, cite individual sentences from the sources using the appropriate citation marker (e.g., [3])."

# # Paraphrased OP generation
# paraphrase_instruction_str = "Respond to the following query by building off of the response below. Specifically, rephrase each sentence in the response using a more fluent and useful wording to convey the same information. In other words, paraphrase each sentence of the response as an improved new sentence, with respect to the query. Do not refer to the response in your revised response. In your response, cite individual sentences from the sources using the appropriate citation marker (e.g., [3])."

# # Entailed OP generation
# entailment_instruction_str = "Respond to the following query by building off of the response below. Specifically, rephrase and combine the sentences in the response by paraphrasing them, cutting out extraneous or redundant information, and simplifying details that are too fine-grained with respect to the question. Also, simplify premises to their logical conclusions to more directly answer the query. Do not refer to the response in your revised response. In your response, cite individual sentences from the sources using the appropriate citation marker (e.g., [3])."

# # Abstractive OP generation
# abstractive_instruction_str = "Respond to the following query by building off of the response below. Specifically, rephrase and combine the sentences in the response by paraphrasing them, cutting out extraneous or redundant information, and simplifying details that are too fine-grained with respect to the question. Also, simplify premises to their logical conclusions to more directly answer the query. Most importantly, use accurate outside information to make the revised response a more useful answer to the query. If the provided response is not an accurate or useful answer to the query, then extensively revise it with accurate outside information. Respond in no more than about 100 words. Do not refer to the response in your revised response. In your response, cite individual sentences from the sources using the appropriate citation marker (e.g., [3])."

In [ ]:
data_batch = [(quoted_op_prompt,
"What is the life cycle of a frog?",
"""
\".S., Applied Ecology, Indiana University BloomingtonB.S., Biology and Chemistry, University of Illinois at Urbana-ChampaignLaura Klappenbach, M.S., is a science writer specializing in ecology, biology, and wildlife.Learn about ourEditorial ProcessUpdated on August 11, 2019The life cycle of a frog consists of three stages: egg, larva, and adult. As the frog grows, it moves through these stages in a process known as metamorphosis.\"
\"But in a few species, parents remain with the eggs to look after them as they develop. As the fertilized eggs mature, the yolk in each egg splits into more and more cells and begins to take the form of a tadpole, the larva of a frog. Within one to three weeks, the egg is ready to hatch, and a tiny tadpole breaks free.03of 04Stage 2: Tadpole (Larva)Johner Images / Getty ImagesTadpoles, frogs' larvae, have rudimentary gills, a mouth, and a long tail. For the first week or two after the tadpole hatches, it moves very little. During this time, the tadpole absorbs the remaining yolk left over from the egg, which provides much-needed nourishment. After absorbing the yolk, the tadpole is strong enough to swim on its own.Most tadpoles feed on algae and other vegetation, so they are considered herbivores. They filter material from the water as they swim or tear away bits of plant material. As the tadpole continues to grow, it begins to develop hind limbs.\"
\"The legless, water-bound tadpoles slowly metamorphose into frogs over the next 14 weeks. First, they grow back legs, then front legs too! Soon after, their body starts to change shape, and they’re able to start eating insects. Next, the tadpoles’ tails shrink away, and skin grows over their gills, as they develop lungs and eardrums! These are super important steps, as they prepare the tadpole for life on land. Once their gills and tails are gone forever, tadpoles undergo one last 24-hour push, where the metamorphosis completes. Once this stage is finished, the baby frogs emerge from the water as tiny adults!\"
"""),
(quoted_op_prompt,
"How many time a day should you check your blood sugar level if you have diabetes?",
"""
\"The frequency of checking your blood sugar depends on your individual needs and the type of diabetes you have. For people with type 1 diabetes, it's recommended to check your blood sugar levels before and after meals, as well as before and after exercise. This helps you monitor how your body responds to food and exercise, and adjust your insulin or diabetes medication as needed.\"
\"For people with type 2 diabetes, the American Diabetes Association recommends checking your blood sugar levels before meals and before bedtime. This helps you adjust your diabetes medication or insulin if your blood sugar levels are too high or too low.\"
\"In some cases, people with diabetes may need to check their blood sugar levels more often, such as when they first start taking medication or insulin. Your healthcare provider will help you determine the best frequency for you.\"
"""),
(quoted_op_prompt,
"Are almonds poisonous for dogs?",
"""
\"There are common foods that dog owners should be cautious about. Grapes and raisins, for instance, can cause severe kidney failure in dogs, even in small amounts. Chocolate is another well-known danger due to theobromine, which dogs metabolize much more slowly than humans. Onions and garlic, whether raw, cooked, or powdered, can damage a dog's red blood cells and lead to anemia. Macadamia nuts are particularly toxic and can cause weakness, vomiting, and tremors in dogs. Almonds are not poisonous for dogs, but they can be harmful if consumed in large quantities. Almonds contain a substance called amygdalin, which can be toxic to dogs in high amounts.\" 

\"If your dog has eaten almonds, monitor their behavior and contact your veterinarian immediately. Additionally, foods containing xylitol, an artificial sweetener found in sugar-free gum and baked goods, can trigger a rapid insulin release leading to hypoglycemia. Avocados contain persin, which can cause vomiting and diarrhea in dogs, and cooked bones can splinter and cause serious internal damage. When in doubt about any human food, it's always best to consult with your veterinarian before sharing it with your furry friend.\"
"""),]

In [88]:
import ast

nq_df = pd.read_csv('autoEval_results/autoEval_gpt4_nq_byQueryOP.csv')
eli5_df = pd.read_csv('autoEval_results/autoEval_gpt4_eli5_nq_byQueryOP.csv')
multihop_df = pd.read_csv('autoEval_results/autoEval_gpt4_multihop_byQueryOP.csv')
mash_df = pd.read_csv('autoEval_results/autoEval_gpt4_mash_byQueryOP.csv')

def get_source_str(value):
    value = ast.literal_eval(value)
    source_str = ""
    for i, source in enumerate(value):
        source_str += f"\"{source}\"\n\n"
    return source_str

def get_numbered_source_str(value):
    source_ls = ast.literal_eval(value)
    source_str = "\n\nSources:\n\n"
    for source in source_ls:
        source_str += "\""
        source = source.strip()
        source_sentence_ls = source.split(".")
        for i, sentence in enumerate(source_sentence_ls):
            
            source_str += f" [{i+1}] {sentence}."
        source_str += "\"\n\n"
    return source_str

# For the quoted OP, add a citation marker for each sentence in the sources.
quoted_sample = nq_df.copy(deep=True).iloc[0:0]
for df in [nq_df, eli5_df, multihop_df, mash_df]:
    quoted_sample = pd.concat([quoted_sample, df[df['op']=='Quoted'].sample(3)])

# TODO add a citation marker before each sentence in the sources.
quoted_sample['Sources String'] = quoted_sample['All Sources'].apply(get_numbered_source_str)
quoted_sample = quoted_sample[['Sources String', 'Question', 'op']]

# For the paraphrased and entailed OPs, use the Used Sources (cited) String column
    # Also, provide the quoted response
paraphrased_and_entailed_sample = nq_df.copy(deep=True).iloc[0:0]
for df in [nq_df, eli5_df, multihop_df, mash_df]:
    paraphrased_and_entailed_sample = pd.concat([paraphrased_and_entailed_sample, df[df['op']=='Paraphrased'].sample(3)])

for df in [nq_df, eli5_df, multihop_df, mash_df]:
    paraphrased_and_entailed_sample = pd.concat([paraphrased_and_entailed_sample, df[df['op']=='Entailed'].sample(3)])

paraphrased_and_entailed_sample['Sources String'] = paraphrased_and_entailed_sample['Used Sources (cited)'].apply(get_source_str)

# Add the quoted response for each query to the front of the Source String
all_df = pd.concat([nq_df, eli5_df, multihop_df, mash_df])
quoted_responses = []
questions = []
for question in paraphrased_and_entailed_sample['Question']:
    quoted_response = all_df[(all_df['Question'] == question) & (all_df['op'] == 'Quoted')]['Output (cited)'].iloc[0]
    quoted_responses.append(quoted_response)
    questions.append(question)

quoted_response_df = pd.DataFrame({'Quoted Response': quoted_responses, 'Question': questions})

paraphrased_and_entailed_sample = pd.merge(paraphrased_and_entailed_sample, quoted_response_df, on='Question', how='left')
paraphrased_and_entailed_sample['Sources String'] = "Initial Response:\n\n"+paraphrased_and_entailed_sample['Quoted Response'] + "\n\nSources:\n\n" + paraphrased_and_entailed_sample['Sources String']
paraphrased_and_entailed_sample = paraphrased_and_entailed_sample[['Sources String', 'Question', 'op']]

training_data = pd.concat([quoted_sample, paraphrased_and_entailed_sample])
training_data = training_data.sample(len(training_data)).reset_index(drop=True)

In [89]:
training_data

,Sources String,Question,op
0,"\n\nSources:\n\n"" [1] Exocrine pancreatic insu...",How is cystic fibrosis treated if you have exo...,Quoted
1,"Initial Response:\n\n[92m""Jack is voiced by C...",who plays jack skellington in nightmare before...,Entailed
2,"Initial Response:\n\n[92m""'It Ain't Me' is a ...",who sings the song it ain't me,Paraphrased
3,"Initial Response:\n\n[92m""You suspect an outb...",When should I call my doctor about shingles?,Paraphrased
4,"\n\nSources:\n\n"" [1] Although procedures vary...",Explain to a third-grader: in the process of s...,Quoted
5,"\n\nSources:\n\n"" [1] Lucian Pintilie (9 Novem...",When did the director of film Ward Six die?,Quoted
6,"\n\nSources:\n\n"" [1] His Majesty, Bunker Bean...","Which film has the director who died earlier, ...",Quoted
7,Initial Response:\n\nThe composer of the film ...,When was the composer of film Une Chambre En V...,Entailed
8,"Initial Response:\n\n[92m""South Sudan was eve...",Explain to a third-grader: when did south suda...,Paraphrased
9,"Initial Response:\n\n[96m""Vellivelichathil is...",Do director of film Vellivelichathil and direc...,Paraphrased


In [ ]:
class QuerySourceDataset(tg.tasks.Dataset):
    def __init__(self):
        nq_df = pd.read_csv('autoEval_results/autoEval_gpt4_nq_byQueryOP.csv')
        eli5_df = pd.read_csv('autoEval_results/autoEval_gpt4_eli5_nq_byQueryOP.csv')
        multihop_df = pd.read_csv('autoEval_results/autoEval_gpt4_multihop_byQueryOP.csv')
        mash_df = pd.read_csv('autoEval_results/autoEval_gpt4_mash_byQueryOP.csv')

        def get_source_str(value):
            value = ast.literal_eval(value)
            source_str = ""
            for i, source in enumerate(value):
                source_str += f"\"{source}\"\n\n"
            return source_str

        def get_numbered_source_str(value):
            source_ls = ast.literal_eval(value)
            source_str = "\n\nSources:\n\n"
            for source in source_ls:
                source_str += "\""
                source = source.strip()
                source_sentence_ls = source.split(".")
                for i, sentence in enumerate(source_sentence_ls):
                    
                    source_str += f" [{i+1}] {sentence}."
                source_str += "\"\n\n"
            return source_str

        # For the quoted OP, add a citation marker for each sentence in the sources.
        quoted_sample = nq_df.copy(deep=True).iloc[0:0]
        for df in [nq_df, eli5_df, multihop_df, mash_df]:
            quoted_sample = pd.concat([quoted_sample, df[df['op']=='Quoted'].sample(3)])

        # Add a citation marker before each sentence in the sources.
        quoted_sample['Sources String'] = quoted_sample['All Sources'].apply(get_numbered_source_str)
        quoted_sample = quoted_sample[['Sources String', 'Question', 'op']]

        # For the paraphrased and entailed OPs, use the Used Sources (cited) String column
            # Also, provide the quoted response
        paraphrased_and_entailed_sample = nq_df.copy(deep=True).iloc[0:0]
        for df in [nq_df, eli5_df, multihop_df, mash_df]:
            paraphrased_and_entailed_sample = pd.concat([paraphrased_and_entailed_sample, df[df['op']=='Paraphrased'].sample(3)])

        for df in [nq_df, eli5_df, multihop_df, mash_df]:
            paraphrased_and_entailed_sample = pd.concat([paraphrased_and_entailed_sample, df[df['op']=='Entailed'].sample(3)])

        paraphrased_and_entailed_sample['Sources String'] = paraphrased_and_entailed_sample['Used Sources (cited)'].apply(get_source_str)

        # Add the quoted response for each query to the front of the Source String
        all_df = pd.concat([nq_df, eli5_df, multihop_df, mash_df])
        quoted_responses = []
        questions = []
        for question in paraphrased_and_entailed_sample['Question']:
            quoted_response = all_df[(all_df['Question'] == question) & (all_df['op'] == 'Quoted')]['Output (cited)'].iloc[0]
            quoted_responses.append(quoted_response)
            questions.append(question)

        quoted_response_df = pd.DataFrame({'Quoted Response': quoted_responses, 'Question': questions})

        paraphrased_and_entailed_sample = pd.merge(paraphrased_and_entailed_sample, quoted_response_df, on='Question', how='left')
        paraphrased_and_entailed_sample['Sources String'] = "Initial Response:\n\n"+paraphrased_and_entailed_sample['Quoted Response'] + "\n\nSources:\n\n" + paraphrased_and_entailed_sample['Sources String']
        paraphrased_and_entailed_sample = paraphrased_and_entailed_sample[['Sources String', 'Question', 'op']]

        training_data = pd.concat([quoted_sample, paraphrased_and_entailed_sample])
        training_data = training_data.sample(len(training_data)).reset_index(drop=True)
        self.training_data = training_data

    def _make_x_i(self, op_prompt, query, sources_str):
        return f"Instructions: {op_prompt}\n\nQuery: {query}\n\nSources: {sources_str}"

    def __getitem__(self, index):
        question = self.training_data.iloc[index]['Question']
        source_str = self.training_data.iloc[index]['Sources String']

        # Quoted OP generation
        op_instruction_dict = {
            "Quoted": "Respond to the following query using word-for-word quotes from the sources provided below. Clearly indicate the quotes with quotation marks to avoid plagiarizing! Be concise in your response and focus on information that responds to the query. Do not refer to the sources in your response. In your response, cite individual sentences from the sources using the appropriate citation marker (e.g., [3]).",
            "Paraphrased":  "Respond to the following query by building off of the response below. Specifically, rephrase each sentence in the response using a more fluent and useful wording to convey the same information. In other words, paraphrase each sentence of the response as an improved new sentence, with respect to the query. Do not refer to the response in your revised response. In your response, cite individual sentences from the sources using the appropriate citation marker (e.g., [3]).",
            "Entailed": "Respond to the following query by building off of the response below. Specifically, rephrase and combine the sentences in the response by paraphrasing them, cutting out extraneous or redundant information, and simplifying details that are too fine-grained with respect to the question. Also, simplify premises to their logical conclusions to more directly answer the query. Do not refer to the response in your revised response. In your response, cite individual sentences from the sources using the appropriate citation marker (e.g., [3])."
            # # Abstractive OP generation
            # abstractive_instruction_str = "Respond to the following query by building off of the response below. Specifically, rephrase and combine the sentences in the response by paraphrasing them, cutting out extraneous or redundant information, and simplifying details that are too fine-grained with respect to the question. Also, simplify premises to their logical conclusions to more directly answer the query. Most importantly, use accurate outside information to make the revised response a more useful answer to the query. If the provided response is not an accurate or useful answer to the query, then extensively revise it with accurate outside information. Respond in no more than about 100 words. Do not refer to the response in your revised response. In your response, cite individual sentences from the sources using the appropriate citation marker (e.g., [3])."
        }
        return self._make_x_i(op_instruction_dict[self.training_data.iloc[index]['op']], question, source_str)

    def __len__(self):
        return len(self.training_data)


In [134]:
# Create the training data
batch_size = 3
training_data = QuerySourceDataset()
train_loader = tg.tasks.DataLoader(training_data, batch_size=batch_size, shuffle=True)    

In [135]:
# Set-up the loss function
loss_system_prompt = tg.Variable(f"""You will evaluate a response to a query, given the goals of following the inner instructions and jointly improving citation precision, citation recall, perceived utility, and fluency. Be super concise. Only identify errors and areas for improvement.

Fluency: {autoeval_fluency_instruction}

Perceived utility: {autoeval_perceived_utility_instruction}

Citation precision: You are an intelligent and fair attribution evaluator. Your task is to verify whether each citation or quote supports at least one claim made in the sentence that contains it. Citations and quotes should not be irrelevant or contradictory to the sentence that contains them.

Citation coverage: You are an intelligent and fair attribution evaluator. Your task is to verify whether the citation(s) or quote(s) cited in each sentence support all of the claims made in that sentence.
""",
    requires_grad=False,
    role_description="system prompt")

loss_fn = tg.TextLoss(loss_system_prompt)

# Set-up the outer prompt
outer_prompt = tg.Variable("""You are an attentive assistant. Follow all instructions carefully. Additionally, provide responses that maximize the following metrics:
- Fluency: The coherence and smoothness of the text when citation markers and quotation marks are removed.
- Perceived utility: How well the generation addresses the user's query, as judged by the user. The generation should be concise and to the point.
- Citation precision: The proportion of citations (or quoted text) that actually support at least one claim in their corresponding sentence (i.e., are relevant and accurate, not irrelevant or contradictory).
- Citation coverage: The proportion of sentences that are fully supported by their cited sources (or quoted text) (i.e., all claims in the sentence can be verified from the citations or quoted text).
""",
            requires_grad=True,
            role_description="system prompt")

# Outer_prompt affects the LLM call
llm_call = tg.autograd.LLMCall(engine=llm_api_eval, system_prompt=outer_prompt)

optimizer = tg.TGD([outer_prompt])

In [138]:
losses = []
for batch in tqdm(train_loader, desc="Training", total=len(training_data)//batch_size):
    for x in batch:
        inner_prompt = tg.Variable(str(x),
                                    requires_grad=False,
                                    role_description="Cited Response Generation Prompt")
        # Solution depends on outer_prompt (via llm_call) and full_prompt
        solution = llm_call(inner_prompt)
        loss = loss_fn(solution)
        losses.append(loss)

    total_loss = tg.sum(losses)

    total_loss.backward()
    optimizer.step()       
    print('~~~~~~~~~~~~~~~~~'*10)
    print("Optimized Outer Prompt: \n", outer_prompt.value)

Training:   8%|▊         | 1/12 [01:40<18:24, 100.42s/it]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Optimized Outer Prompt: 
 You are an attentive assistant. Follow all instructions carefully. Additionally, maximize these metrics in your responses:

- **Fluency:** Ensure text is coherent, concise, and idiomatic, with smooth flow and clear logical connections. Avoid ambiguous or unidiomatic modifiers, and restructure sentences for clarity, reducing redundancy and grouping related content logically.
- **Perceived Utility:** Directly and efficiently address the user's query. Present key takeaways, organize advice or facts into skimmable, logically-sequenced sections, and make all factual referents (such as names, places, or entities) explicit—never rely on ambiguous pronouns or vague summaries.
- **Citation Precision:** Only use numbered citations or quoted text for claims explicitly supported, word-for-word, by the pr

Training:  17%|█▋        | 2/12 [04:39<24:24, 146.46s/it]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Optimized Outer Prompt: 
 You are an attentive assistant. Follow all instructions carefully. Provide responses that maximize the following metrics and adhere to these operational requirements:

**Fluency:** Craft responses that are coherent, idiomatic, and concise, ensuring smooth logical connections and clear referents. Avoid ambiguous or vague language (e.g., pronouns with unclear antecedents), unidiomatic phrasing, and redundancy. Use varied sentence structure and logical grouping of related content, especially when presenting lists or instructions.

**Perceived Utility:** Directly address the user's query using clear, skimmable, logically-sequenced sections. Explicitly name all entities and facts to maintain clarity—never rely on ambiguous pronouns or references. Include rationales for advice, highlight key takeaw

Training:  25%|██▌       | 3/12 [08:53<29:19, 195.55s/it]

...

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Optimized Outer Prompt: 
 You are an attentive assistant. Follow all instructions carefully. For every query, response, or revision—regardless of style or complexity—meticulously maximize the following metrics and obey these foundational operational rules:

**Citation Precision at the Atomic Level:**  
For every unique factual claim, data point, or procedural step, place a citation marker *immediately after* the precise claim it supports—never after an entire sentence, paragraph, or group of facts unless the source directly covers all elements verbatim. Do not cite a source for a claim it merely implies, partially covers, or is only contextually related to; split or restructure sentences so each atomic claim is individually cited. Grouped or bundled citation markers (e.g., [1][2]) at the end of multi-claim sentences a

Training:  92%|█████████▏| 11/12 [1:39:24<13:31, 811.12s/it]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Optimized Outer Prompt: 
 You are an attentive assistant. Follow all instructions carefully. For every query, response, or revision—regardless of style, context, or complexity—rigorously maximize all of the following metrics and strictly adhere to every operational requirement:

**Atomic Citation-Claim Mapping:**  
For every factual claim, action, or step—no matter how minor—ensure that a citation marker is placed *immediately after* (and only after) the precise claim or data point it supports. Never cite a source for a claim it only implies, partially covers, or is related to by context; split or restructure sentences so each atomic claim is individually and unambiguously cited. Strictly prohibit citation bundling/clustering at the end of compound or multi-fact sentences unless every citation independently and verbat

Training: 100%|██████████| 12/12 [1:56:58<00:00, 584.88s/it]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Optimized Outer Prompt: 
 You are an attentive assistant. Follow all instructions carefully. For every response—regardless of task, input style, or complexity—rigorously maximize all of the following metrics and comply precisely with every operational requirement below. Treat atomic, unambiguous citation-claim mapping as a non-negotiable baseline for every claim, instruction, or revision. At every step, explicitly decompose, audit, and verify each atomic factual claim, detail, or data point for precise, one-to-one citation support, and transparently flag any inference or use of background knowledge.

**Citation Precision and Atomic Mapping:**  
- For every distinct factual claim, data point, or procedural step, place a citation marker *immediately after* (not at sentence ends or for claim clusters) the precise claim, 

# Verify performance improved